In [1]:
# # Cell 0) (Optional) Install dependencies
# # Run this only if pyblip is not installed.

# !pip install pyblip cvxpy scikit-learn pandas numpy
# !pip install tqdm

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# !pip -q install pyblip tqdm scikit-learn pandas numpy

In [4]:
from pathlib import Path
import os, random
PROJECT_DIR = Path("/content/drive/MyDrive/BLiP")
DATA_DIR = PROJECT_DIR / "data"
OUT_DIR = PROJECT_DIR / "out"

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_DIR:", DATA_DIR)
print("OUT_DIR:", OUT_DIR)

PROJECT_DIR: /content/drive/MyDrive/BLiP
DATA_DIR: /content/drive/MyDrive/BLiP/data
OUT_DIR: /content/drive/MyDrive/BLiP/out


In [5]:
# Cell 1) Imports + Paths + Config

from __future__ import annotations

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
os.environ.setdefault("PYTHONHASHSEED", str(RANDOM_STATE))

# BLiP / Bayesian settings
Q_FDR = 0.2     # target FDR level
N_SAMPLES = 1400     # posterior samples per chain (increase later)
N_CHAINS = 6          # number of MCMC chains
BATCH_SIZE = 3        # minibatch size used by the sampler
Burn = 100

# RF settings (final model)
RF_N_ESTIMATORS = 300
RF_MAX_DEPTH = None

# Preprocessing toggle
USE_STANDARDIZE = True  # Fit on TRAIN only, apply to TEST (no leakage)

# Paths: assuming notebook is in BLiP/code/


X_TRAIN_PATH = DATA_DIR / "X_train.csv"
X_TEST_PATH  = DATA_DIR / "X_test.csv"
Y_TRAIN_PATH = DATA_DIR / "y_train.csv"
Y_TEST_PATH  = DATA_DIR / "y_test.csv"

print("DATA_DIR:", DATA_DIR)
print("Files exist:",
      X_TRAIN_PATH.exists(), X_TEST_PATH.exists(),
      Y_TRAIN_PATH.exists(), Y_TEST_PATH.exists())

DATA_DIR: /content/drive/MyDrive/BLiP/data
Files exist: True True True True


In [6]:
# Cell 2) Load data (GBM) + sanity checks

X_train_df = pd.read_csv(X_TRAIN_PATH)
X_test_df  = pd.read_csv(X_TEST_PATH)

y_train_df = pd.read_csv(Y_TRAIN_PATH)
y_test_df  = pd.read_csv(Y_TEST_PATH)

if y_train_df.shape[1] != 1 or y_test_df.shape[1] != 1:
    raise ValueError(
        f"Expected y_train/y_test to have exactly 1 column, "
        f"got y_train={y_train_df.shape}, y_test={y_test_df.shape}"
    )

y_train = y_train_df.iloc[:, 0].to_numpy().ravel().astype(int)
y_test  = y_test_df.iloc[:, 0].to_numpy().ravel().astype(int)

# Keep feature names
feature_names = list(X_train_df.columns)

# Convert to numpy
X_train = X_train_df.to_numpy(dtype=float)
X_test  = X_test_df.to_numpy(dtype=float)


# --- Force dtypes to match pyblip's Cython expectations (Windows-safe) ---

# y must be int32 / C-int (avoid int64)
y_train = np.asarray(y_train, dtype=np.intc)   # np.intc = C int (usually int32 on Windows)
y_test  = np.asarray(y_test,  dtype=np.intc)

# X must be float64 and C-contiguous
X_train = np.asarray(X_train, dtype=np.float64, order="C")
X_test  = np.asarray(X_test,  dtype=np.float64, order="C")

print("Dtypes check:")
print("  X_train:", X_train.dtype, "C_CONTIGUOUS:", X_train.flags["C_CONTIGUOUS"])
print("  y_train:", y_train.dtype)

print("=== Shapes ===")
print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_test :", X_test.shape,  "y_test :", y_test.shape)

print("\n=== Label check ===")
print("Train positive rate:", y_train.mean())
print("Test  positive rate:", y_test.mean())

# Basic consistency checks
if X_train.shape[1] != X_test.shape[1]:
    raise ValueError("Train/test feature dimension mismatch.")
if len(feature_names) != X_train.shape[1]:
    raise ValueError("Feature name length mismatch.")

Dtypes check:
  X_train: float64 C_CONTIGUOUS: True
  y_train: int32
=== Shapes ===
X_train: (82, 724) y_train: (82,)
X_test : (36, 724) y_test : (36,)

=== Label check ===
Train positive rate: 0.6585365853658537
Test  positive rate: 0.6666666666666666


In [7]:
# Preprocessing toggle
USE_STANDARDIZE = True  # Fit on TRAIN only, apply to TEST (no leakage)

In [8]:
# Cell 3) Preprocess (fit on TRAIN only) -> transform TEST (temporal-safe / no leakage)

if USE_STANDARDIZE:
    scaler = StandardScaler(with_mean=True, with_std=True)
    X_train_pp = scaler.fit_transform(X_train)   # fit on TRAIN only
    X_test_pp  = scaler.transform(X_test)        # transform TEST only
else:
    X_train_pp = X_train.copy()
    X_test_pp  = X_test.copy()
# After preprocessing
X_train_pp = np.asarray(X_train_pp, dtype=np.float64, order="C")
X_test_pp  = np.asarray(X_test_pp,  dtype=np.float64, order="C")

print("USE_STANDARDIZE =", USE_STANDARDIZE)
print("X_train_pp mean (first 5 cols):", X_train_pp[:, :5].mean(axis=0))
print("X_train_pp std  (first 5 cols):", X_train_pp[:, :5].std(axis=0))

USE_STANDARDIZE = True
X_train_pp mean (first 5 cols): [-5.31417728e-17  6.76965259e-17 -4.06179155e-17  2.57246798e-17
 -8.39436921e-17]
X_train_pp std  (first 5 cols): [1. 1. 1. 1. 1.]


In [10]:
# Cell 4) Baseline RF (all features) - evaluate once on TEST

rf_base = RandomForestClassifier(
    n_estimators=RF_N_ESTIMATORS,
    max_depth=RF_MAX_DEPTH,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    class_weight="balanced",
)

rf_base.fit(X_train_pp, y_train)

base_proba = rf_base.predict_proba(X_test_pp)[:, 1]
base_pred  = (base_proba >= 0.5).astype(int)

print("=== Baseline RF (all features) ===")
print("Accuracy:", accuracy_score(y_test, base_pred))
try:
    print("ROC AUC :", roc_auc_score(y_test, base_proba))
except ValueError:
    print("ROC AUC : undefined (only one class present in y_test)")
print("Confusion matrix:\n", confusion_matrix(y_test, base_pred))

=== Baseline RF (all features) ===
Accuracy: 0.6944444444444444
ROC AUC : 0.7031250000000001
Confusion matrix:
 [[ 3  9]
 [ 2 22]]


In [ ]:
import numpy as np
import pyblip
from pyblip.probit import ProbitSpikeSlab

BETAS_PATH = OUT_DIR / "pm_betas.npy"

if BETAS_PATH.exists():
    print("Found saved betas:", BETAS_PATH)
    betas = np.load(BETAS_PATH)
else:
    pm = ProbitSpikeSlab(
        X=np.asarray(X_train_pp, dtype=np.float64, order="C"),
        y=np.asarray(y_train, dtype=np.intc),
    )

    pm.sample(
        N=N_SAMPLES,
        chains=N_CHAINS,
        bsize=BATCH_SIZE,
        burn=Burn,
    )

    betas = np.asarray(pm.betas)
    np.save(BETAS_PATH, betas)
    print("Saved betas to:", BETAS_PATH)

print("Posterior betas shape:", betas.shape)

In [ ]:
# Cell 6) Run BLiP (FDR control) -> detections (groups) + selected feature indices

import numpy as np
from pyblip.blip import BLiP

# ✅ always use the loaded/saved betas (works whether pm exists or not)
# betas shape should be (chains, N, p) or similar; np.load gives the same array back.

np.random.seed(RANDOM_STATE)

detections = BLiP(
    samples=betas,          # <-- FIX: was pm.betas
    q=Q_FDR,
    error="fdr",
    verbose=False,
    solver="CLARABEL",
    perturb=False,
    random_state=RANDOM_STATE,
)

print("=== BLiP results ===")
print("q (FDR target):", Q_FDR)
print("#detections (groups):", len(detections))

selected_idx = set()
for d in detections:
    grp = getattr(d, "group", None)
    if grp is None:
        continue
    selected_idx |= set(grp)

selected_idx_sorted = sorted(selected_idx)
selected_features = [feature_names[i] for i in selected_idx_sorted]

print("#selected features (union of groups):", len(selected_features))

for k, d in enumerate(detections, start=1):
    grp = sorted(list(getattr(d, "group", [])))
    preview_idx = grp[:10]
    preview_names = [feature_names[i] for i in preview_idx]
    more = "" if len(grp) <= 10 else f" ... (+{len(grp)-10} more)"
    print(f"Det {k}: size={len(grp)} idx={preview_idx}{more}")
    print(f"      features={preview_names}{more}")

if len(selected_idx_sorted) == 0:
    print("\nWARNING: BLiP selected 0 features. Falling back to all features.")
    selected_idx_sorted = list(range(X_train_pp.shape[1]))
    selected_features = feature_names

In [ ]:
# ============================================================
# RF Hyperparameter Tuning Settings (EXACT same as your old code)
# - PARAM_DIST
# - STAGE1_N_ITER
# - cv5
# - tune_rf_for_features()
# ============================================================

from scipy.stats import randint
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV

STAGE1_N_ITER = globals().get("STAGE1_N_ITER", 80)  # keep 80 if defined, same as before
RANDOM_STATE  = 42

# EXACT same CV as your old code
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=2042)

# EXACT same parameter distribution as your old code
PARAM_DIST = {
    "n_estimators": randint(100, 650),
    "max_depth": [None, 8, 12],
    "max_features": ["sqrt", "log2", 0.3, 0.5, 0.7],
    "min_samples_split": randint(2, 20),
    "min_samples_leaf": randint(1, 8),
    "bootstrap": [True],
    "max_samples": [None, 0.6, 0.8, 1.0],
    "class_weight": [None, "balanced"],
    "criterion": ["gini", "entropy"],
}

def tune_rf_for_features(X_sub, y, label="RF"):
    """
    Tune RF on a given design matrix X_sub (numpy array) and label vector y,
    using EXACT RandomizedSearchCV settings as your old code.
    """
    rs = RandomizedSearchCV(
        estimator=RandomForestClassifier(random_state=2042, n_jobs=-1),
        param_distributions=PARAM_DIST,
        n_iter=STAGE1_N_ITER,
        scoring="roc_auc",
        cv=cv5,
        n_jobs=-1,
        random_state=2042,
        verbose=0,
    )
    rs.fit(X_sub, y)

    best_params = dict(rs.best_params_)
    # cleanup for direct RF init (same logic)
    best_params.pop("random_state", None)
    best_params.pop("n_jobs", None)
    if best_params.get("bootstrap") is False:
        best_params.pop("max_samples", None)

    print(f"[CV] {label} best AUC = {rs.best_score_:.4f}")
    print(f"[CV] {label} best params = {best_params}")
    return best_params, float(rs.best_score_)

In [ ]:
# --- Hyperparameter tuning (same setting) -> refit -> proba ---

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from scipy.stats import randint
import numpy as np

# EXACT tuning setup
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=2042)
PARAM_DIST = {
    "n_estimators": randint(100, 650),
    "max_depth": [None, 8, 12],
    "max_features": ["sqrt", "log2", 0.3, 0.5, 0.7],
    "min_samples_split": randint(2, 20),
    "min_samples_leaf": randint(1, 8),
    "bootstrap": [True],
    "max_samples": [None, 0.6, 0.8, 1.0],
    "class_weight": [None, "balanced"],
    "criterion": ["gini", "entropy"],
}
N_ITER = 5

X_train_sel = X_train_pp[:, selected_idx_sorted]
X_test_sel  = X_test_pp[:, selected_idx_sorted]

rs = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=2042, n_jobs=-1),
    param_distributions=PARAM_DIST,
    n_iter=N_ITER,
    scoring="roc_auc",
    cv=cv5,
    n_jobs=-1,
    random_state=2042,
    verbose=0
)
rs.fit(X_train_sel, y_train)

best_params = dict(rs.best_params_)
best_params.pop("random_state", None)
best_params.pop("n_jobs", None)
if best_params.get("bootstrap") is False:
    best_params.pop("max_samples", None)

print("[CV] best AUC:", rs.best_score_)
print("[CV] best params:", best_params)

# refit with tuned hyperparams
rf_final = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1, **best_params)
rf_final.fit(X_train_sel, y_train)

train_proba = rf_final.predict_proba(X_train_sel)[:, 1]
test_proba  = rf_final.predict_proba(X_test_sel)[:, 1]

In [ ]:
# Cell 7) Binary plots (TRAIN/TEST): ROC + Confusion Matrix + Precision/Recall bars
#         -> save as PNG/PDF under OUT_DIR (or FIG_DIR)

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.metrics import (
    roc_curve, roc_auc_score,
    confusion_matrix,
    precision_score, recall_score
)

# where to save
FIG_DIR = OUT_DIR / "figure"
FIG_DIR.mkdir(parents=True, exist_ok=True)

def plot_binary_panel(y_true, proba, tag, threshold=0.5, save_dir=FIG_DIR):
    # metrics
    auc = roc_auc_score(y_true, proba)
    fpr, tpr, _ = roc_curve(y_true, proba)

    pred = (proba >= threshold).astype(int)
    cm = confusion_matrix(y_true, pred)  # [[TN, FP],[FN, TP]]
    prec = precision_score(y_true, pred, zero_division=0)
    rec  = recall_score(y_true, pred, zero_division=0)

    # figure (1 row x 3 cols)
    fig, axes = plt.subplots(1, 3, figsize=(15.2, 4.6))

    # (A) ROC
    ax = axes[0]
    ax.plot(fpr, tpr, lw=2.2, label=f"AUC={auc:.3f}")
    ax.plot([0, 1], [0, 1], ls=":", lw=1.2, label="Chance")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(f"ROC | {tag}")
    ax.legend(frameon=False, loc="lower right")

    # (B) Confusion matrix
    ax = axes[1]
    im = ax.imshow(cm, interpolation="nearest")
    ax.set_title(f"Confusion Matrix | {tag}")
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(["0", "1"]); ax.set_yticklabels(["0", "1"])
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    # (C) Precision/Recall bars
    ax = axes[2]
    ax.bar(["Precision", "Recall"], [prec, rec])
    ax.set_ylim(0, 1.0)
    ax.set_title(f"Precision/Recall | {tag}")
    ax.text(0, min(1.0, prec + 0.03), f"{prec:.3f}", ha="center", va="bottom")
    ax.text(1, min(1.0, rec  + 0.03), f"{rec:.3f}",  ha="center", va="bottom")

    fig.tight_layout()

    # save
    stem = f"blip_rf_binary_{tag.replace(' ', '_').replace('|','').replace('(', '').replace(')', '')}"
    out_png = save_dir / f"{stem}.png"
    out_pdf = save_dir / f"{stem}.pdf"
    fig.savefig(out_png, dpi=300, bbox_inches="tight")
    fig.savefig(out_pdf, bbox_inches="tight")

    plt.show()
    print("[SAVED]", out_png)
    print("[SAVED]", out_pdf)

# ---- TRAIN ----
plot_binary_panel(
    y_true=y_train,
    proba=train_proba,
    tag="TRAIN",
    threshold=0.5,
)

# ---- TEST ----
plot_binary_panel(
    y_true=y_test,
    proba=test_proba,
    tag="TEST",
    threshold=0.5,
)